# TP N°3 : Classification avec Naive Bayes

**Module :** FD1 — **Section :** M2 Bioinfo

## Objectif

Comparer deux algorithmes de classification basés sur le théorème de Bayes, **GaussianNB** et **MultinomialNB**, sur la détection de spams SMS.

## Dataset

Le dataset est le **SMS Spam Collection** (UCI). Il contient 5 574 messages étiquetés `ham` (légitime) ou `spam`, soit 4 827 ham et 747 spam.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay
)

## 1. Chargement du dataset

Le fichier `SMSSpamCollection` contient une ligne par message : le label (`ham` ou `spam`), une tabulation, puis le texte. On convertit le label en classe numérique : spam = 1 (positif), ham = 0 (négatif).

In [5]:
df = pd.read_csv("SMSSpamCollection", sep="\t", header=None, names=["label", "message"])
df["class"] = (df["label"] == "spam").astype(int)

print("Nombre de messages :", len(df))
print(df["label"].value_counts())
df.head()

Nombre de messages : 5572
label
ham     4825
spam     747
Name: count, dtype: int64


,label,message,class
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## 2. Division apprentissage / test

On réserve 20 % des messages pour le test. L'option `stratify=y` conserve la même proportion de spams dans les deux ensembles, ce qui rend la comparaison équitable.

In [6]:
X = df["message"]
y = df["class"]

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7, stratify=y
)

print("Apprentissage :", len(X_train_txt), "messages")
print("Test :", len(X_test_txt), "messages")

Apprentissage : 4457 messages
Test : 1115 messages


## 3. Prétraitement : texte vers comptages de mots

Les algorithmes travaillent sur des nombres, pas sur du texte. `CountVectorizer` transforme chaque message en vecteur contenant le nombre d'occurrences de chaque mot. Le vocabulaire est appris **uniquement sur l'apprentissage**, pour éviter que le modèle ne voie des mots du test.

In [7]:
vectorizer = CountVectorizer(max_features=1000, stop_words="english")

X_train = vectorizer.fit_transform(X_train_txt)
X_test = vectorizer.transform(X_test_txt)

print("Taille de la matrice d'apprentissage :", X_train.shape)

Taille de la matrice d'apprentissage : (4457, 1000)


## 4. Entraînement des modèles

- **GaussianNB** suppose que chaque variable suit une loi normale dans chaque classe. Il ne prend pas en charge les matrices creuses : on convertit donc la matrice en format dense avec `toarray()`.
- **MultinomialNB** est conçu pour des comptages (nombres d'occurrences). Il accepte directement la matrice creuse.

In [8]:
gnb = GaussianNB().fit(X_train.toarray(), y_train)
y_pred_gnb = gnb.predict(X_test.toarray())

mnb = MultinomialNB().fit(X_train, y_train)
y_pred_mnb = mnb.predict(X_test)

## 5. Matrice de confusion et mesures

La matrice de confusion compte quatre cas : vrais négatifs (VN), faux positifs (FP), faux négatifs (FN) et vrais positifs (VP). Les mesures se calculent à partir de ces valeurs :

- **Accuracy** = (VP + VN) / total
- **Precision** = VP / (VP + FP)
- **Recall** = VP / (VP + FN)
- **F1-score** = 2 × précision × rappel / (précision + rappel)

On les calcule d'abord à la main, puis avec scikit-learn pour vérifier.

In [9]:
def matrice_confusion_manuelle(y_vrai, y_pred):
    y_vrai = np.array(y_vrai)
    y_pred = np.array(y_pred)

    VP = np.sum((y_vrai == 1) & (y_pred == 1))
    VN = np.sum((y_vrai == 0) & (y_pred == 0))
    FP = np.sum((y_vrai == 0) & (y_pred == 1))
    FN = np.sum((y_vrai == 1) & (y_pred == 0))

    return VN, FP, FN, VP


def mesures_manuelles(y_vrai, y_pred):
    VN, FP, FN, VP = matrice_confusion_manuelle(y_vrai, y_pred)

    accuracy = (VP + VN) / (VP + VN + FP + FN)
    precision = VP / (VP + FP) if (VP + FP) > 0 else 0.0
    recall = VP / (VP + FN) if (VP + FN) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {"Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-score": f1}


def mesures_sklearn(y_vrai, y_pred):
    return {
        "Accuracy": accuracy_score(y_vrai, y_pred),
        "Precision": precision_score(y_vrai, y_pred),
        "Recall": recall_score(y_vrai, y_pred),
        "F1-score": f1_score(y_vrai, y_pred)
    }

In [10]:
manuel = pd.DataFrame({
    "GaussianNB": mesures_manuelles(y_test, y_pred_gnb),
    "MultinomialNB": mesures_manuelles(y_test, y_pred_mnb)
})

sk = pd.DataFrame({
    "GaussianNB": mesures_sklearn(y_test, y_pred_gnb),
    "MultinomialNB": mesures_sklearn(y_test, y_pred_mnb)
})

print("=== Mesures calculées à la main ===")
print(manuel.round(3))
print("\n=== Mesures scikit-learn ===")
print(sk.round(3))
print("\nMêmes résultats (manuel = sklearn) ?", np.allclose(manuel, sk))

=== Mesures calculées à la main ===
           GaussianNB  MultinomialNB
Accuracy        0.764          0.984
Precision       0.355          0.946
Recall          0.933          0.933
F1-score        0.514          0.939

=== Mesures scikit-learn ===
           GaussianNB  MultinomialNB
Accuracy        0.764          0.984
Precision       0.355          0.946
Recall          0.933          0.933
F1-score        0.514          0.939

Mêmes résultats (manuel = sklearn) ? True
